In [10]:
with open("2.sh", "w") as doc:
    for edge in ["hard_ratio"]:
        for low_k in [0.1, 0.2, 0.3, 0.4]:
            for high_k in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
                for combine_x in ["", "--combine_x"]:
                    for split in ["official", "simple"]:
                        doc.write("python HLCL_ssl.py --device 1 --dataset amazon-ratings --low_k {} --high_k {} {} --edge hard_ratio --split {} \n".format(low_k, high_k, combine_x, split))


In [1]:
print(str(None))

None


In [567]:
with open("grid_search_squirrel_3.sh", "w") as doc:
    for edge in ["soft"]:
        for low_k in [0]:
            for high_k in [0]:
                for combine_x in ["", "--combine_x"]:
                    for two_hop in ["", "--two_hop"]:
                        for intraview_negs in ["", "--intraview_negs", ""]:
                            for md in ["union", "low", "high"]:
                                for num_layer in [2, 3, 4]:
                                    doc.write("python HLCL_hard.py --device 7 --dataset squirrel --pre_learning_rate 0.001 --aug1 0.2 --aug2 0.2 --runs 5 --per_epoch 50 --low_k {} --high_k {} {} {} --edge {} --md {} {} --num_layer {} \n".format(low_k, high_k, combine_x, two_hop, edge, md, intraview_negs, num_layer))


In [1]:
import torch
j = 1

/home/hyang/anaconda3/envs/clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork
from torch_geometric import transforms as T
file_loc = './data/'
dataset_name = "chameleon"
# dataset = Planetoid(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
# dataset = WebKB(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
dataset = WikipediaNetwork(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
data = dataset[0]

/home/hyang/anaconda3/envs/clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

: 

In [28]:
from torch_geometric.data import Data

In [39]:
import numpy as np

a = np.load("/home/hyang/HL_Contrast/data/chameleon_filtered.npz")
data = Data(x = torch.tensor(a["node_features"]), edge_index = torch.tensor(a["edges"].T), y = torch.tensor(a["node_labels"]))
data.train_mask = torch.tensor(a["train_masks"]).t()
data.val_mask = torch.tensor(a["val_masks"]).t()
data.test_mask = torch.tensor(a["test_masks"]).t()

In [46]:
data.train_mask[0]

tensor([False, False,  True,  True,  True, False,  True,  True, False,  True])

In [41]:
data.y.shape

torch.Size([890])

In [12]:
from utility.data import dataset_split
import scipy
import torch
import numpy as np
from sklearn.preprocessing import label_binarize
mat = scipy.io.loadmat("/home/hyang/HL_Contrast/data/facebook100/Penn94.mat")
A = mat['A']
metadata = mat['local_info']

In [14]:
adj = torch.tensor(A.todense())

In [15]:
from torch_geometric.utils import dense_to_sparse

edge = dense_to_sparse(adj)

In [16]:
metadata = int(metadata)
label = metadata[:, 1] - 1  # gender label, -1 means unlabeled

# make features into one-hot encodings
feature_vals = np.hstack(
    (np.expand_dims(metadata[:, 0], 1), metadata[:, 2:]))
features = np.empty((A.shape[0], 0))
for col in range(feature_vals.shape[1]):
    feat_col = feature_vals[:, col]
    feat_onehot = label_binarize(feat_col, classes=np.unique(feat_col))
    features = np.hstack((features, feat_onehot))

node_feat = torch.tensor(features, dtype=torch.float)

/tmp/ipykernel_178402/1273907029.py:1: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  metadata = metadata.astype(np.int)


In [1]:
from utility.data import dataset_split

In [2]:
data = dataset_split(dataset_name="Penn94")

mission complete


In [5]:
from torch_geometric.utils import is_undirected

In [6]:
is_undirected(data.edge_index)

True